# Supervised ResNet50 baseline for culvert blockage classification

This notebook documents the supervised ResNet50 reference baseline used in the dissertation.

ResNet50 is initialised with ImageNet pretrained weights. The convolutional backbone is frozen and only the final fully connected classification layer is trained. This provides a conventional supervised transfer-learning baseline against which the dissertation's foundation-model approaches can be compared.

> **Reproducibility note.** Machine-specific Google Drive and Colab paths have been removed from this public version. Execution outputs have also been cleared so that no local directory information is retained. The code expects repository-relative inputs under `data/` and writes generated artefacts to `outputs/resnet50_supervised/`.

In [ ]:
# Environment, reproducibility and repository paths

from pathlib import Path
import os
import time
import random
import copy
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve
)

# A fixed seed makes the site-level validation split and model initialisation
# reproducible across runs, subject to the usual limits of GPU determinism.
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Repository-relative paths.
# Expected public-repository layout:
#   data/cropped_dataset_index.csv
#   data/site_split.csv
#   data/cropped_images/...
#   outputs/resnet50_supervised/...
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
IMAGE_ROOT = DATA_DIR / "cropped_images"
EXPERIMENT_FOLDER = PROJECT_ROOT / "outputs" / "resnet50_supervised"

BASELINE_FOLDER = EXPERIMENT_FOLDER / "baseline"
TUNING_FOLDER = EXPERIMENT_FOLDER / "tuning"
FINAL_FOLDER = EXPERIMENT_FOLDER / "final"

for folder in [EXPERIMENT_FOLDER, BASELINE_FOLDER, TUNING_FOLDER, FINAL_FOLDER]:
    folder.mkdir(parents=True, exist_ok=True)

CROPPED_INDEX_PATH = DATA_DIR / "cropped_dataset_index.csv"
SPLIT_PATH = DATA_DIR / "site_split.csv"

print("Repository root:", PROJECT_ROOT)
print("Dataset index:", CROPPED_INDEX_PATH)
print("Fixed site split:", SPLIT_PATH)
print("Output directory:", EXPERIMENT_FOLDER)


In [ ]:
# Verify required repository inputs

required_files = [CROPPED_INDEX_PATH, SPLIT_PATH]
missing_inputs = [str(path) for path in required_files if not path.exists()]

if missing_inputs:
    raise FileNotFoundError(
        "Required input files were not found. Expected repository-relative files:\n"
        + "\n".join(missing_inputs)
    )

print("Required metadata files found.")


In [ ]:
# Load the cropped-image index

dataset_df = pd.read_csv(CROPPED_INDEX_PATH)

required_columns = {"site", "label", "filename", "cropped_path"}
assert required_columns.issubset(dataset_df.columns), (
    f"Dataset index must contain: {sorted(required_columns)}"
)
assert len(dataset_df) == 4000, "Expected the fixed 4,000-image study dataset."

print("Number of images:", len(dataset_df))
display(dataset_df.head())


In [ ]:
# Resolve image paths without machine-specific directories

# The original experimental index stored absolute Colab/Google Drive paths.
# For public release, only the image filename and site identifier are used to
# resolve files under the repository's data/cropped_images directory.
# Both a flat image directory and site-specific subdirectories are supported.
def resolve_public_image_path(row):
    filename = Path(str(row["filename"])).name
    candidates = [
        IMAGE_ROOT / filename,
        IMAGE_ROOT / str(row["site"]) / filename,
    ]

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    # Return the canonical flat-layout path so the verification below gives a
    # clear missing-file error rather than silently falling back to a private path.
    return str(candidates[0])


dataset_df["local_path"] = dataset_df.apply(resolve_public_image_path, axis=1)
dataset_df["file_exists"] = dataset_df["local_path"].map(os.path.exists)

print("Existing cropped images:", int(dataset_df["file_exists"].sum()))
print("Missing cropped images:", int((~dataset_df["file_exists"]).sum()))

assert dataset_df["file_exists"].all(), (
    "Some cropped images are missing from data/cropped_images. "
    "Check the repository data layout before running the experiment."
)


## 1. Cross-site experimental split

Within the development set, two sites are selected reproducibly for validation using the fixed random seed; the remaining six sites are used for model fitting during configuration experiments. No held-out test image contributes to model selection, hyperparameter tuning or threshold selection.

In [ ]:
# Cell 5: Load fixed held-out site split

split_summary = pd.read_csv(
    SPLIT_PATH
)

DEVELOPMENT_SITES = (
    split_summary.loc[
        split_summary["role"]
        == "development",
        "site"
    ]
    .tolist()
)

TEST_SITES = (
    split_summary.loc[
        split_summary["role"]
        == "test",
        "site"
    ]
    .tolist()
)

assert len(DEVELOPMENT_SITES) == 8
assert len(TEST_SITES) == 2

# The final evaluation sites were fixed before model development.
EXPECTED_TEST_SITES = {
    "Cornwall_BudeCedarGrove",
    "sites_brutondam_cam1",
}
assert set(TEST_SITES) == EXPECTED_TEST_SITES, (
    "site_split.csv does not contain the fixed held-out test sites used in the study."
)

assert set(
    DEVELOPMENT_SITES
).isdisjoint(
    TEST_SITES
)

assert set(
    dataset_df["site"].unique()
) == set(
    DEVELOPMENT_SITES + TEST_SITES
)

print("Development sites:")

for site in DEVELOPMENT_SITES:
    print(f"  {site}")

print("\nHeld-out test sites:")

for site in TEST_SITES:
    print(f"  {site}")

In [ ]:
# Cell 6: Split development sites into training and validation sites

split_rng = np.random.default_rng(
    RANDOM_SEED
)

VALIDATION_SITES = sorted(
    split_rng.choice(
        DEVELOPMENT_SITES,
        size=2,
        replace=False
    ).tolist()
)

TRAIN_SITES = [
    site
    for site in DEVELOPMENT_SITES
    if site not in VALIDATION_SITES
]

assert len(TRAIN_SITES) == 6
assert len(VALIDATION_SITES) == 2

print("Training sites:")

for site in TRAIN_SITES:
    print(f"  {site}")

print("\nValidation sites:")

for site in VALIDATION_SITES:
    print(f"  {site}")

print("\nHeld-out test sites:")

for site in TEST_SITES:
    print(f"  {site}")

In [ ]:
# Cell 7: Create training, validation and test DataFrames

train_df = (
    dataset_df[
        dataset_df["site"].isin(
            TRAIN_SITES
        )
    ]
    .reset_index(drop=True)
)

validation_df = (
    dataset_df[
        dataset_df["site"].isin(
            VALIDATION_SITES
        )
    ]
    .reset_index(drop=True)
)

test_df = (
    dataset_df[
        dataset_df["site"].isin(
            TEST_SITES
        )
    ]
    .reset_index(drop=True)
)

print(
    "Training images:",
    len(train_df)
)

print(
    "Validation images:",
    len(validation_df)
)

print(
    "Test images:",
    len(test_df)
)

print("\nTraining class counts:")
display(
    train_df["label"].value_counts()
)

print("\nValidation class counts:")
display(
    validation_df["label"].value_counts()
)

print("\nTest class counts:")
display(
    test_df["label"].value_counts()
)

## 2. Image preprocessing and dataset construction

Images are resized to 224 × 224 pixels and normalised using the ImageNet channel statistics associated with the pretrained ResNet50 weights. The deterministic preprocessing used for validation and test data ensures that performance comparisons are not affected by stochastic transformations.

Labels are encoded as `0 = clear` and `1 = blocked`. Data loaders shuffle only the training data; validation and test order remain fixed.

In [ ]:
# Cell 8: R0 baseline preprocessing

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]

R0_TRANSFORM = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print(
    R0_TRANSFORM
)

In [ ]:
# Cell 9: Dataset class

class CulvertResNetDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform


    def __len__(self):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        index
    ):

        row = self.dataframe.iloc[
            index
        ]

        image = Image.open(
            row["local_path"]
        ).convert("RGB")

        if self.transform is not None:

            image = self.transform(
                image
            )

        label = (
            1
            if row["label"] == "blocked"
            else 0
        )

        return (
            image,
            label,
            row["site"],
            row["local_path"]
        )

In [ ]:
# Cell 10: R0 DataLoaders

BATCH_SIZE = 32

train_dataset = CulvertResNetDataset(
    train_df,
    transform=R0_TRANSFORM
)

validation_dataset = CulvertResNetDataset(
    validation_df,
    transform=R0_TRANSFORM
)

test_dataset = CulvertResNetDataset(
    test_df,
    transform=R0_TRANSFORM
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

## 3. R0: pretrained ResNet50 reference configuration

The initial model uses an ImageNet-pretrained ResNet50 as a **frozen feature extractor**. Freezing the backbone means that the pretrained convolutional parameters are not updated on the culvert dataset. The original 1,000-class ImageNet classification layer is replaced by a two-output linear layer, and only this task-specific layer is optimised.

This design isolates the value of transferring generic visual features while limiting the number of trainable parameters. Batch-normalisation layers in the frozen backbone are kept in evaluation mode so that their pretrained running statistics are not inadvertently updated during training.

R0 uses cross-entropy loss and stochastic gradient descent (SGD) with learning rate 0.001 and momentum 0.9. The best development checkpoint is selected by validation F1 score.

In [ ]:
# Cell 11: R0 baseline ResNet50

weights = ResNet50_Weights.IMAGENET1K_V2

r0_model = models.resnet50(
    weights=weights
)

# Freeze pretrained backbone
for parameter in r0_model.parameters():
    parameter.requires_grad = False

# Replace ImageNet classifier
in_features = r0_model.fc.in_features

r0_model.fc = nn.Linear(
    in_features,
    2
)

r0_model = r0_model.to(
    DEVICE
)

total_parameters = sum(
    parameter.numel()
    for parameter in r0_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in r0_model.parameters()
    if parameter.requires_grad
)

print("Model: ResNet50")
print("Weights: ImageNet pretrained")
print("Frozen: convolutional backbone")
print("Trainable: final fully connected layer")

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

print(
    "Trainable percentage:",
    f"{100 * trainable_parameters / total_parameters:.4f}%"
)

In [ ]:
# Cell 12: R0 preset training configuration

from torch.optim import lr_scheduler

R0_LEARNING_RATE = 0.001
R0_MOMENTUM = 0.9

r0_criterion = nn.CrossEntropyLoss()

r0_optimizer = optim.SGD(
    r0_model.fc.parameters(),
    lr=R0_LEARNING_RATE,
    momentum=R0_MOMENTUM
)

r0_scheduler = lr_scheduler.StepLR(
    r0_optimizer,
    step_size=7,
    gamma=0.1
)

print("R0 configuration")
print("Optimizer: SGD")
print("Learning rate:", R0_LEARNING_RATE)
print("Momentum:", R0_MOMENTUM)
print("Weight decay: 0")
print("Training augmentation: none")
print("Loss: CrossEntropyLoss")

In [ ]:
# Cell 13: Shared training and validation functions

def set_frozen_batchnorm_eval(
    model
):
    """
    Keep BatchNorm running statistics fixed when the
    pretrained ResNet backbone is frozen.

    model.train() normally places BatchNorm layers in
    training mode even when requires_grad=False.
    This function keeps their stored ImageNet statistics fixed.
    """

    for module in model.modules():

        if isinstance(
            module,
            nn.BatchNorm2d
        ):
            module.eval()


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):

    model.train()

    # Frozen-backbone transfer learning:
    # keep pretrained BatchNorm statistics fixed.
    set_frozen_batchnorm_eval(
        model
    )

    running_loss = 0.0
    all_targets = []
    all_predictions = []

    for images, labels, _, _ in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad()

        logits = model(
            images
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()
        optimizer.step()

        running_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_targets.extend(
            labels.detach().cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )

    epoch_f1 = f1_score(
        all_targets,
        all_predictions,
        zero_division=0
    )

    return (
        epoch_loss,
        epoch_f1
    )


def evaluate_model(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0
    all_targets = []
    all_predictions = []
    all_scores = []

    with torch.no_grad():

        for images, labels, _, _ in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            logits = model(
                images
            )

            loss = criterion(
                logits,
                labels
            )

            running_loss += (
                loss.item()
                * images.size(0)
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            # Standard 0.5-equivalent decision used during
            # development comparisons.
            predictions = torch.argmax(
                logits,
                dim=1
            )

            all_targets.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_scores.extend(
                probabilities.cpu().numpy()
            )

    metrics = {

        "loss":
            running_loss
            / len(loader.dataset),

        "auc":
            roc_auc_score(
                all_targets,
                all_scores
            ),

        "accuracy":
            accuracy_score(
                all_targets,
                all_predictions
            ),

        "precision":
            precision_score(
                all_targets,
                all_predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                all_targets,
                all_predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                all_targets,
                all_predictions,
                zero_division=0
            )
    }

    return (
        metrics,
        np.array(all_targets),
        np.array(all_predictions),
        np.array(all_scores)
    )

In [ ]:
# Cell 14: R0 one-epoch runtime probe

probe_model = copy.deepcopy(
    r0_model
)

probe_optimizer = optim.SGD(
    probe_model.fc.parameters(),
    lr=R0_LEARNING_RATE,
    momentum=R0_MOMENTUM
)

start_time = time.time()

probe_train_loss, probe_train_f1 = (
    train_one_epoch(
        probe_model,
        train_loader,
        r0_criterion,
        probe_optimizer
    )
)

probe_validation_metrics, _, _, _ = (
    evaluate_model(
        probe_model,
        validation_loader,
        r0_criterion
    )
)

probe_seconds = (
    time.time()
    - start_time
)

print(
    "One-epoch runtime:",
    f"{probe_seconds / 60:.2f} minutes"
)

print(
    "Training loss:",
    f"{probe_train_loss:.4f}"
)

print(
    "Training F1:",
    f"{probe_train_f1:.4f}"
)

print(
    "Validation AUC:",
    f"{probe_validation_metrics['auc']:.4f}"
)

print(
    "Validation F1:",
    f"{probe_validation_metrics['f1']:.4f}"
)

In [ ]:
# Cell 15: Train R0 baseline ResNet50

# Reference:
# Optimizer follows the PyTorch transfer-learning tutorial:
# SGD, learning rate 0.001, momentum 0.9,
# StepLR(step_size=7, gamma=0.1).
#
# Experimental choice:
# MAX_EPOCHS = 20 is a fixed computational budget for this study.
# The best checkpoint is selected using validation F1,

R0_MAX_EPOCHS = 20

r0_history = []

best_r0_f1 = -np.inf
best_r0_epoch = None
best_r0_state = None

start_time = time.time()


for epoch in range(
    1,
    R0_MAX_EPOCHS + 1
):

    train_loss, train_f1 = train_one_epoch(
        r0_model,
        train_loader,
        r0_criterion,
        r0_optimizer
    )


    validation_metrics, _, _, _ = evaluate_model(
        r0_model,
        validation_loader,
        r0_criterion
    )


    current_lr = (
        r0_optimizer
        .param_groups[0]["lr"]
    )


    r0_history.append({

        "epoch":
            epoch,

        "learning_rate":
            current_lr,

        "train_loss":
            train_loss,

        "train_f1":
            train_f1,

        "validation_loss":
            validation_metrics["loss"],

        "validation_auc":
            validation_metrics["auc"],

        "validation_accuracy":
            validation_metrics["accuracy"],

        "validation_precision":
            validation_metrics["precision"],

        "validation_recall":
            validation_metrics["recall"],

        "validation_f1":
            validation_metrics["f1"]
    })


    print(
        f"Epoch {epoch:02d} | "
        f"lr {current_lr:.6f} | "
        f"train F1 {train_f1:.4f} | "
        f"val AUC {validation_metrics['auc']:.4f} | "
        f"val F1 {validation_metrics['f1']:.4f}"
    )


    # Save the checkpoint with the highest validation F1
    if (
        validation_metrics["f1"]
        > best_r0_f1
    ):

        best_r0_f1 = (
            validation_metrics["f1"]
        )

        best_r0_epoch = epoch

        best_r0_state = copy.deepcopy(
            r0_model.state_dict()
        )


    # PyTorch transfer-learning tutorial scheduler
    r0_scheduler.step()


r0_training_seconds = (
    time.time()
    - start_time
)


# Restore best validation-F1 checkpoint
r0_model.load_state_dict(
    best_r0_state
)


r0_history_df = pd.DataFrame(
    r0_history
)


print("\nR0 complete")

print(
    "Best validation F1:",
    f"{best_r0_f1:.4f}"
)

print(
    "Best epoch:",
    best_r0_epoch
)

print(
    "Training time:",
    f"{r0_training_seconds / 60:.2f} minutes"
)

In [ ]:
# Cell 16: R0 learning history

display(
    r0_history_df
)


fig, axis = plt.subplots(
    figsize=(8, 5)
)

axis.plot(
    r0_history_df["epoch"],
    r0_history_df["train_f1"],
    marker="o",
    label="Training F1"
)

axis.plot(
    r0_history_df["epoch"],
    r0_history_df["validation_f1"],
    marker="o",
    label="Validation F1"
)

axis.set_xlabel(
    "Epoch"
)

axis.set_ylabel(
    "F1 score"
)

axis.set_title(
    "R0 ResNet50 baseline training"
)

axis.set_ylim(
    0,
    1
)

axis.legend()

fig.tight_layout()

plt.show()

### R0 baseline interpretation

The R0 baseline showed rapid improvement on the training data, with training F1 increasing from approximately 0.70 to above 0.90. Validation F1, however, remained comparatively stable at approximately 0.73–0.76 throughout training.

The increasing separation between training and validation performance indicates that the classifier increasingly fitted the training sites without producing a corresponding improvement on the unseen validation sites. Validation performance reached its strongest values early in training and remained broadly stable thereafter.

This baseline therefore provides evidence that pretrained ResNet50 features contain useful information for blockage classification, while also showing limited cross-site improvement from continued training of the classification head alone. The R0 result is retained as the reference condition against which subsequent changes to training augmentation, optimiser and weight decay are evaluated.

In [ ]:
# Cell 17: Save R0 development result

best_r0_row = (
    r0_history_df.loc[
        r0_history_df[
            "validation_f1"
        ].idxmax()
    ]
)


r0_summary = pd.DataFrame([
    {
        "experiment":
            "R0",

        "model":
            "ResNet50",

        "pretrained_weights":
            "ImageNet1K_V2",

        "trainable_component":
            "fc",

        "augmentation":
            "none",

        "optimizer":
            "SGD",

        "learning_rate":
            R0_LEARNING_RATE,

        "momentum":
            R0_MOMENTUM,

        "weight_decay":
            0.0,

        "best_epoch":
            int(
                best_r0_epoch
            ),

        "validation_auc":
            float(
                best_r0_row[
                    "validation_auc"
                ]
            ),

        "validation_accuracy":
            float(
                best_r0_row[
                    "validation_accuracy"
                ]
            ),

        "validation_precision":
            float(
                best_r0_row[
                    "validation_precision"
                ]
            ),

        "validation_recall":
            float(
                best_r0_row[
                    "validation_recall"
                ]
            ),

        "validation_f1":
            float(
                best_r0_row[
                    "validation_f1"
                ]
            ),

        "training_minutes":
            r0_training_seconds
            / 60
    }
])


display(
    r0_summary
)


r0_history_df.to_csv(
    EXPERIMENT_FOLDER / "R0_training_history.csv",
    index=False
)

r0_summary.to_csv(
    EXPERIMENT_FOLDER / "R0_validation_summary.csv",
    index=False
)

torch.save(
    best_r0_state,
    EXPERIMENT_FOLDER / "R0_best_model_state.pt"
)

print(
    "R0 outputs saved."
)

## 4. Controlled development experiments

Model configuration is developed incrementally so that the effect of each intervention can be interpreted against the preceding configuration rather than changing several factors simultaneously.

**R1** introduces mild training-only augmentation while retaining deterministic validation preprocessing. **R2** changes the optimiser from SGD to Adam. **R3** evaluates weight decay as an explicit regularisation parameter. The same site-level training and validation split is retained throughout these comparisons, allowing changes in validation performance to be attributed more directly to the configuration being tested.

In [ ]:
# Cell 18: R1 training augmentation

R1_TRAIN_TRANSFORM = transforms.Compose([

    # Experimental intervention:
    # mild training-only augmentation to increase
    # appearance variation while preserving screen content.
    transforms.RandomResizedCrop(
        224,
        scale=(0.90, 1.00)
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),

    transforms.ToTensor(),

    # ImageNet normalisation required by the
    # pretrained torchvision ResNet50 weights.
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# Validation remains deterministic and identical to R0
R1_VALIDATION_TRANSFORM = R0_TRANSFORM


r1_train_dataset = CulvertResNetDataset(
    train_df,
    transform=R1_TRAIN_TRANSFORM
)

r1_validation_dataset = CulvertResNetDataset(
    validation_df,
    transform=R1_VALIDATION_TRANSFORM
)


r1_train_loader = DataLoader(
    r1_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

r1_validation_loader = DataLoader(
    r1_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("R1 change from R0: training augmentation ON")
print("Optimizer: unchanged (SGD)")
print("Weight decay: unchanged (0)")
print("Validation preprocessing: unchanged")

In [ ]:
# Cell 19: Build R1 ResNet50 model

weights = ResNet50_Weights.IMAGENET1K_V2

r1_model = models.resnet50(
    weights=weights
)

# Same frozen-backbone strategy as R0
for parameter in r1_model.parameters():
    parameter.requires_grad = False

in_features = r1_model.fc.in_features

r1_model.fc = nn.Linear(
    in_features,
    2
)

r1_model = r1_model.to(
    DEVICE
)

print("R1 model ready")
print("Difference from R0: training augmentation only")

In [ ]:
# Cell 20: R1 training configuration

# Same PyTorch transfer-learning optimizer recipe as R0.
# Only augmentation changes in R1.

r1_criterion = nn.CrossEntropyLoss()

r1_optimizer = optim.SGD(
    r1_model.fc.parameters(),
    lr=R0_LEARNING_RATE,
    momentum=R0_MOMENTUM
)

r1_scheduler = lr_scheduler.StepLR(
    r1_optimizer,
    step_size=7,
    gamma=0.1
)

R1_MAX_EPOCHS = 20

print("R1 configuration")
print("Optimizer: SGD")
print("Learning rate:", R0_LEARNING_RATE)
print("Momentum:", R0_MOMENTUM)
print("Weight decay: 0")
print("Training augmentation: ON")
print("Validation preprocessing: same as R0")

In [ ]:
# Cell 21: Train R1 with augmentation

r1_history = []

best_r1_f1 = -np.inf
best_r1_epoch = None
best_r1_state = None

start_time = time.time()


for epoch in range(
    1,
    R1_MAX_EPOCHS + 1
):

    train_loss, train_f1 = train_one_epoch(
        r1_model,
        r1_train_loader,
        r1_criterion,
        r1_optimizer
    )

    validation_metrics, _, _, _ = evaluate_model(
        r1_model,
        r1_validation_loader,
        r1_criterion
    )

    current_lr = (
        r1_optimizer
        .param_groups[0]["lr"]
    )

    r1_history.append({

        "epoch": epoch,
        "learning_rate": current_lr,
        "train_loss": train_loss,
        "train_f1": train_f1,

        "validation_loss":
            validation_metrics["loss"],

        "validation_auc":
            validation_metrics["auc"],

        "validation_accuracy":
            validation_metrics["accuracy"],

        "validation_precision":
            validation_metrics["precision"],

        "validation_recall":
            validation_metrics["recall"],

        "validation_f1":
            validation_metrics["f1"]
    })


    print(
        f"Epoch {epoch:02d} | "
        f"lr {current_lr:.6f} | "
        f"train F1 {train_f1:.4f} | "
        f"val AUC {validation_metrics['auc']:.4f} | "
        f"val F1 {validation_metrics['f1']:.4f}"
    )


    if (
        validation_metrics["f1"]
        > best_r1_f1
    ):

        best_r1_f1 = (
            validation_metrics["f1"]
        )

        best_r1_epoch = epoch

        best_r1_state = copy.deepcopy(
            r1_model.state_dict()
        )


    r1_scheduler.step()


r1_training_seconds = (
    time.time()
    - start_time
)


r1_model.load_state_dict(
    best_r1_state
)


r1_history_df = pd.DataFrame(
    r1_history
)


print("\nR1 complete")

print(
    "Best validation F1:",
    f"{best_r1_f1:.4f}"
)

print(
    "Best epoch:",
    best_r1_epoch
)

print(
    "Training time:",
    f"{r1_training_seconds / 60:.2f} minutes"
)

In [ ]:
# Cell 22: Compare R0 baseline against R1 augmentation

r0_best = r0_history_df.loc[
    r0_history_df[
        "validation_f1"
    ].idxmax()
]

r1_best = r1_history_df.loc[
    r1_history_df[
        "validation_f1"
    ].idxmax()
]


r0_r1_comparison = pd.DataFrame([
    {
        "experiment":
            "R0",

        "augmentation":
            "none",

        "optimizer":
            "SGD",

        "weight_decay":
            0.0,

        "best_epoch":
            int(
                r0_best["epoch"]
            ),

        "validation_auc":
            r0_best[
                "validation_auc"
            ],

        "validation_f1":
            r0_best[
                "validation_f1"
            ]
    },

    {
        "experiment":
            "R1",

        "augmentation":
            "on",

        "optimizer":
            "SGD",

        "weight_decay":
            0.0,

        "best_epoch":
            int(
                r1_best["epoch"]
            ),

        "validation_auc":
            r1_best[
                "validation_auc"
            ],

        "validation_f1":
            r1_best[
                "validation_f1"
            ]
    }
])


display(
    r0_r1_comparison
)

In [ ]:
# Cell 23: Build R2 ResNet50 and Adam configuration

# R2 changes one experimental factor relative to R1:
# optimizer: SGD -> Adam
#
# Unchanged:
# pretrained ResNet50 weights
# frozen backbone
# trainable fc layer
# R1 training augmentation
# validation preprocessing
# weight decay = 0
# 20-epoch training budget
# checkpoint selection by validation F1

weights = ResNet50_Weights.IMAGENET1K_V2

r2_model = models.resnet50(
    weights=weights
)

for parameter in r2_model.parameters():
    parameter.requires_grad = False

in_features = r2_model.fc.in_features

r2_model.fc = nn.Linear(
    in_features,
    2
)

r2_model = r2_model.to(
    DEVICE
)


r2_criterion = nn.CrossEntropyLoss()


# PyTorch Adam optimizer.
#
# Adam's PyTorch default learning rate is 0.001.
# We retain that default here rather than tuning
# learning rate at this experimental stage.
r2_optimizer = optim.Adam(
    r2_model.fc.parameters(),
    lr=0.001,
    weight_decay=0.0
)


R2_LEARNING_RATE = 0.001
R2_MAX_EPOCHS = 20


print("R2 configuration")
print("Model: ResNet50")
print("Trainable component: fc")
print("Training augmentation: ON")
print("Optimizer: Adam")
print("Learning rate:", R2_LEARNING_RATE)
print("Weight decay: 0")
print("Scheduler: none")
print("Epoch budget:", R2_MAX_EPOCHS)

In [ ]:
# Cell 24: Train R2 with Adam

r2_history = []

best_r2_f1 = -np.inf
best_r2_epoch = None
best_r2_state = None

start_time = time.time()


for epoch in range(
    1,
    R2_MAX_EPOCHS + 1
):

    train_loss, train_f1 = train_one_epoch(
        r2_model,
        r1_train_loader,
        r2_criterion,
        r2_optimizer
    )

    validation_metrics, _, _, _ = evaluate_model(
        r2_model,
        r1_validation_loader,
        r2_criterion
    )


    r2_history.append({

        "epoch":
            epoch,

        "learning_rate":
            R2_LEARNING_RATE,

        "train_loss":
            train_loss,

        "train_f1":
            train_f1,

        "validation_loss":
            validation_metrics["loss"],

        "validation_auc":
            validation_metrics["auc"],

        "validation_accuracy":
            validation_metrics["accuracy"],

        "validation_precision":
            validation_metrics["precision"],

        "validation_recall":
            validation_metrics["recall"],

        "validation_f1":
            validation_metrics["f1"]
    })


    print(
        f"Epoch {epoch:02d} | "
        f"train F1 {train_f1:.4f} | "
        f"val AUC {validation_metrics['auc']:.4f} | "
        f"val F1 {validation_metrics['f1']:.4f}"
    )


    # Experimental selection rule:
    # retain the epoch with highest validation F1.
    if (
        validation_metrics["f1"]
        > best_r2_f1
    ):

        best_r2_f1 = (
            validation_metrics["f1"]
        )

        best_r2_epoch = epoch

        best_r2_state = copy.deepcopy(
            r2_model.state_dict()
        )


r2_training_seconds = (
    time.time()
    - start_time
)


r2_model.load_state_dict(
    best_r2_state
)


r2_history_df = pd.DataFrame(
    r2_history
)


print("\nR2 complete")

print(
    "Best validation F1:",
    f"{best_r2_f1:.4f}"
)

print(
    "Best epoch:",
    best_r2_epoch
)

print(
    "Training time:",
    f"{r2_training_seconds / 60:.2f} minutes"
)

In [ ]:
# Cell 25: Compare R0, R1 and R2

r2_best = r2_history_df.loc[
    r2_history_df[
        "validation_f1"
    ].idxmax()
]


r0_r1_r2_comparison = pd.DataFrame([

    {
        "experiment": "R0",
        "augmentation": "none",
        "optimizer": "SGD",
        "weight_decay": 0.0,
        "best_epoch": int(r0_best["epoch"]),
        "validation_auc": r0_best["validation_auc"],
        "validation_f1": r0_best["validation_f1"]
    },

    {
        "experiment": "R1",
        "augmentation": "on",
        "optimizer": "SGD",
        "weight_decay": 0.0,
        "best_epoch": int(r1_best["epoch"]),
        "validation_auc": r1_best["validation_auc"],
        "validation_f1": r1_best["validation_f1"]
    },

    {
        "experiment": "R2",
        "augmentation": "on",
        "optimizer": "Adam",
        "weight_decay": 0.0,
        "best_epoch": int(r2_best["epoch"]),
        "validation_auc": r2_best["validation_auc"],
        "validation_f1": r2_best["validation_f1"]
    }

])


display(
    r0_r1_r2_comparison
)

In [ ]:
# Cell 26: R3 weight-decay search

# Experimental objective:
# Test whether L2 weight decay improves the R2 Adam configuration.
#
# Fixed:
# - ResNet50 ImageNet pretrained weights
# - frozen backbone
# - trainable fc layer only
# - R1 augmentation
# - Adam optimizer
# - learning rate = 0.001
# - 20 epochs
# - selection by highest validation F1
#
# Search variable:
# - weight decay
#
# PyTorch reference:
# torch.optim.Adam exposes weight_decay as an optimizer parameter.
# 0.0 is included as the no-regularisation control.

R3_WEIGHT_DECAYS = [
    0.0,
    1e-5,
    1e-4,
    1e-3
]

r3_results = []
r3_histories = {}
r3_states = {}


for weight_decay in R3_WEIGHT_DECAYS:

    print(
        "\nWeight decay:",
        weight_decay
    )

    # Fresh pretrained model for every configuration
    model = models.resnet50(
        weights=ResNet50_Weights.IMAGENET1K_V2
    )

    for parameter in model.parameters():
        parameter.requires_grad = False

    in_features = model.fc.in_features

    model.fc = nn.Linear(
        in_features,
        2
    )

    model = model.to(
        DEVICE
    )


    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.fc.parameters(),
        lr=R2_LEARNING_RATE,
        weight_decay=weight_decay
    )


    history = []

    best_f1 = -np.inf
    best_epoch = None
    best_state = None


    for epoch in range(
        1,
        R2_MAX_EPOCHS + 1
    ):

        train_loss, train_f1 = train_one_epoch(
            model,
            r1_train_loader,
            criterion,
            optimizer
        )

        validation_metrics, _, _, _ = evaluate_model(
            model,
            r1_validation_loader,
            criterion
        )


        history.append({

            "epoch":
                epoch,

            "weight_decay":
                weight_decay,

            "train_loss":
                train_loss,

            "train_f1":
                train_f1,

            "validation_loss":
                validation_metrics["loss"],

            "validation_auc":
                validation_metrics["auc"],

            "validation_accuracy":
                validation_metrics["accuracy"],

            "validation_precision":
                validation_metrics["precision"],

            "validation_recall":
                validation_metrics["recall"],

            "validation_f1":
                validation_metrics["f1"]
        })


        if (
            validation_metrics["f1"]
            > best_f1
        ):

            best_f1 = (
                validation_metrics["f1"]
            )

            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )


    history_df = pd.DataFrame(
        history
    )

    r3_histories[
        weight_decay
    ] = history_df

    r3_states[
        weight_decay
    ] = best_state


    best_row = history_df.loc[
        history_df[
            "validation_f1"
        ].idxmax()
    ]


    r3_results.append({

        "weight_decay":
            weight_decay,

        "best_epoch":
            int(best_epoch),

        "validation_auc":
            best_row[
                "validation_auc"
            ],

        "validation_accuracy":
            best_row[
                "validation_accuracy"
            ],

        "validation_precision":
            best_row[
                "validation_precision"
            ],

        "validation_recall":
            best_row[
                "validation_recall"
            ],

        "validation_f1":
            best_row[
                "validation_f1"
            ]
    })


    print(
        f"Best epoch: {best_epoch} | "
        f"AUC: {best_row['validation_auc']:.4f} | "
        f"F1: {best_row['validation_f1']:.4f}"
    )


r3_results_df = pd.DataFrame(
    r3_results
).sort_values(
    "validation_f1",
    ascending=False
).reset_index(
    drop=True
)


print("\nR3 weight-decay ranking:")

display(
    r3_results_df
)

## 5. Hyperparameter search with Optuna

After the controlled comparisons, Optuna is used to refine the selected optimisation configuration on the **development data only**. The objective is validation F1 score. Each trial starts from a fresh pretrained ResNet50 with the same frozen-backbone design, preventing parameter states from leaking between trials.

The held-out test sites remain inaccessible during this search. This separation is essential: using test performance to choose hyperparameters would turn the test set into an additional validation set and invalidate the final estimate of cross-site generalisation.

In [ ]:
# Cell 27: Optuna setup

!pip -q install optuna

import optuna

print(
    "Optuna version:",
    optuna.__version__
)

In [ ]:
# Cell 28: Optuna objective for ResNet50

OPTUNA_MAX_EPOCHS = 25

def objective(trial):

    optimizer_name = trial.suggest_categorical(
        "optimizer",
        [
            "Adam",
            "AdamW"
        ]
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        1e-3,
        log=True
    )

    use_augmentation = trial.suggest_categorical(
        "augmentation",
        [
            False,
            True
        ]
    )


    if use_augmentation:

        trial_train_loader = r1_train_loader

    else:

        trial_train_loader = train_loader


    trial_validation_loader = validation_loader


    model = models.resnet50(
        weights=ResNet50_Weights.IMAGENET1K_V2
    )

    for parameter in model.parameters():
        parameter.requires_grad = False

    in_features = model.fc.in_features

    model.fc = nn.Linear(
        in_features,
        2
    )

    model = model.to(
        DEVICE
    )


    criterion = nn.CrossEntropyLoss()


    # PyTorch reference:
    # Adam default learning rate = 0.001.
    # AdamW default learning rate = 0.001.
    # Learning rate is not tuned in this study,
    # following supervisor feedback to limit compute.

    if optimizer_name == "Adam":

        optimizer = optim.Adam(
            model.fc.parameters(),
            lr=0.001,
            weight_decay=weight_decay
        )

    else:

        optimizer = optim.AdamW(
            model.fc.parameters(),
            lr=0.001,
            weight_decay=weight_decay
        )


    best_validation_f1 = -np.inf


    for epoch in range(
        1,
        OPTUNA_MAX_EPOCHS + 1
    ):

        train_one_epoch(
            model,
            trial_train_loader,
            criterion,
            optimizer
        )


        validation_metrics, _, _, _ = (
            evaluate_model(
                model,
                trial_validation_loader,
                criterion
            )
        )


        current_f1 = validation_metrics[
            "f1"
        ]


        best_validation_f1 = max(
            best_validation_f1,
            current_f1
        )


        # Report intermediate result to Optuna
        trial.report(
            current_f1,
            step=epoch
        )


        # Allow poor trials to stop early
        if trial.should_prune():

            raise optuna.TrialPruned()


    return float(
        best_validation_f1
    )

In [ ]:
# Cell 29: Run Optuna search

sampler = optuna.samplers.TPESampler(
    seed=RANDOM_SEED
)

pruner = optuna.pruners.MedianPruner(
    n_startup_trials=3,
    n_warmup_steps=5
)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="resnet50_optimizer_weight_decay_augmentation"
)


start_time = time.time()


study.optimize(
    objective,
    n_trials=8
)


optuna_seconds = (
    time.time()
    - start_time
)


print(
    "Optuna search complete."
)

print(
    "Search time:",
    f"{optuna_seconds / 60:.2f} minutes"
)

print(
    "Best validation F1:",
    f"{study.best_value:.4f}"
)

print(
    "Best parameters:"
)

print(
    study.best_params
)

In [ ]:
# Cell 30: Optuna trial results

optuna_results_df = study.trials_dataframe(
    attrs=(
        "number",
        "value",
        "params",
        "state"
    )
)


display(
    optuna_results_df
)


optuna_results_df.to_csv(
    EXPERIMENT_FOLDER / "R4_optuna_trials.csv",
    index=False
)

In [ ]:
# Cell 31: Extract selected Optuna configuration and best epoch

best_trial = study.best_trial

best_intermediate_values = (
    best_trial.intermediate_values
)

BEST_OPTUNA_EPOCH = max(
    best_intermediate_values,
    key=best_intermediate_values.get
)

BEST_OPTUNA_F1 = (
    best_intermediate_values[
        BEST_OPTUNA_EPOCH
    ]
)

FINAL_OPTIMIZER = (
    best_trial.params["optimizer"]
)

FINAL_WEIGHT_DECAY = (
    best_trial.params["weight_decay"]
)

FINAL_AUGMENTATION = (
    best_trial.params["augmentation"]
)

print("Selected ResNet50 configuration")
print("Trial:", best_trial.number)
print("Optimizer:", FINAL_OPTIMIZER)
print("Weight decay:", FINAL_WEIGHT_DECAY)
print("Augmentation:", FINAL_AUGMENTATION)
print("Best epoch:", BEST_OPTUNA_EPOCH)
print(
    "Best validation F1:",
    f"{BEST_OPTUNA_F1:.4f}"
)

In [ ]:
# Cell 32: Rebuild corrected R4 ResNet50 configuration

r4_model = models.resnet50(
    weights=ResNet50_Weights.IMAGENET1K_V2
)

for parameter in r4_model.parameters():
    parameter.requires_grad = False

in_features = r4_model.fc.in_features

r4_model.fc = nn.Linear(
    in_features,
    2
)

r4_model = r4_model.to(
    DEVICE
)

r4_criterion = nn.CrossEntropyLoss()

r4_optimizer = optim.AdamW(
    r4_model.fc.parameters(),
    lr=0.001,
    weight_decay=FINAL_WEIGHT_DECAY
)

R4_EPOCHS = BEST_OPTUNA_EPOCH

# Optuna selected augmentation = False
r4_train_loader = train_loader
r4_validation_loader = validation_loader

print("R4 selected configuration")
print("Optimizer:", FINAL_OPTIMIZER)
print("Learning rate: 0.001")
print("Weight decay:", FINAL_WEIGHT_DECAY)
print("Augmentation:", FINAL_AUGMENTATION)
print("Training epochs:", R4_EPOCHS)

In [ ]:
# Cell 33: Train corrected R4 configuration

r4_history = []

start_time = time.time()

for epoch in range(
    1,
    R4_EPOCHS + 1
):

    train_loss, train_f1 = train_one_epoch(
        r4_model,
        r4_train_loader,
        r4_criterion,
        r4_optimizer
    )

    validation_metrics, _, _, _ = evaluate_model(
        r4_model,
        r4_validation_loader,
        r4_criterion
    )

    r4_history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_f1":
            train_f1,

        "validation_loss":
            validation_metrics["loss"],

        "validation_auc":
            validation_metrics["auc"],

        "validation_accuracy":
            validation_metrics["accuracy"],

        "validation_precision":
            validation_metrics["precision"],

        "validation_recall":
            validation_metrics["recall"],

        "validation_f1":
            validation_metrics["f1"]
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train F1 {train_f1:.4f} | "
        f"val AUC {validation_metrics['auc']:.4f} | "
        f"val F1 {validation_metrics['f1']:.4f}"
    )


r4_training_seconds = (
    time.time()
    - start_time
)

r4_history_df = pd.DataFrame(
    r4_history
)

r4_selected_row = (
    r4_history_df.loc[
        r4_history_df["epoch"]
        == R4_EPOCHS
    ]
    .iloc[0]
)

print("\nR4 reconstruction complete")

print(
    "Validation AUC:",
    f"{r4_selected_row['validation_auc']:.4f}"
)

print(
    "Validation F1:",
    f"{r4_selected_row['validation_f1']:.4f}"
)

print(
    "Training time:",
    f"{r4_training_seconds / 60:.2f} minutes"
)

In [ ]:
# Cell 33a: Select final classification threshold on development validation data

# Threshold selection uses development validation data only.
# The held-out test sites remain untouched.

(
    r4_validation_metrics,
    r4_validation_targets,
    _,
    r4_validation_scores
) = evaluate_model(
    r4_model,
    r4_validation_loader,
    r4_criterion
)


threshold_candidates = np.linspace(
    0.0,
    1.0,
    1001
)

threshold_rows = []


for threshold in threshold_candidates:

    predictions = (
        r4_validation_scores
        >= threshold
    ).astype(int)

    threshold_rows.append({

        "threshold":
            threshold,

        "accuracy":
            accuracy_score(
                r4_validation_targets,
                predictions
            ),

        "precision":
            precision_score(
                r4_validation_targets,
                predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                r4_validation_targets,
                predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                r4_validation_targets,
                predictions,
                zero_division=0
            )
    })


threshold_results_df = pd.DataFrame(
    threshold_rows
)


best_threshold_row = (
    threshold_results_df.loc[
        threshold_results_df[
            "f1"
        ].idxmax()
    ]
)


FINAL_CLASSIFICATION_THRESHOLD = float(
    best_threshold_row[
        "threshold"
    ]
)


print(
    "Selected classification threshold:",
    f"{FINAL_CLASSIFICATION_THRESHOLD:.3f}"
)

print(
    "Validation F1 at selected threshold:",
    f"{best_threshold_row['f1']:.4f}"
)

print(
    "Validation precision:",
    f"{best_threshold_row['precision']:.4f}"
)

print(
    "Validation recall:",
    f"{best_threshold_row['recall']:.4f}"
)


threshold_results_df.to_csv(
    EXPERIMENT_FOLDER / "ResNet50_validation_threshold_search.csv",
    index=False
)

In [ ]:
# Cell 34: Final corrected ResNet50 development comparison

# All results below come from the corrected frozen-backbone
# implementation in which BatchNorm running statistics remain fixed.

best_r3 = r3_results_df.iloc[0]


resnet50_development_comparison = pd.DataFrame([

    {
        "experiment":
            "R0",

        "change":
            "Preset baseline",

        "optimizer":
            "SGD",

        "augmentation":
            False,

        "weight_decay":
            0.0,

        "best_epoch":
            int(
                r0_best["epoch"]
            ),

        "validation_auc":
            float(
                r0_best["validation_auc"]
            ),

        "validation_f1":
            float(
                r0_best["validation_f1"]
            )
    },

    {
        "experiment":
            "R1",

        "change":
            "Training augmentation",

        "optimizer":
            "SGD",

        "augmentation":
            True,

        "weight_decay":
            0.0,

        "best_epoch":
            int(
                r1_best["epoch"]
            ),

        "validation_auc":
            float(
                r1_best["validation_auc"]
            ),

        "validation_f1":
            float(
                r1_best["validation_f1"]
            )
    },

    {
        "experiment":
            "R2",

        "change":
            "Adam optimisation recipe",

        "optimizer":
            "Adam",

        "augmentation":
            True,

        "weight_decay":
            0.0,

        "best_epoch":
            int(
                r2_best["epoch"]
            ),

        "validation_auc":
            float(
                r2_best["validation_auc"]
            ),

        "validation_f1":
            float(
                r2_best["validation_f1"]
            )
    },

    {
        "experiment":
            "R3",

        "change":
            "Weight decay search",

        "optimizer":
            "Adam",

        "augmentation":
            True,

        "weight_decay":
            float(
                best_r3["weight_decay"]
            ),

        "best_epoch":
            int(
                best_r3["best_epoch"]
            ),

        "validation_auc":
            float(
                best_r3["validation_auc"]
            ),

        "validation_f1":
            float(
                best_r3["validation_f1"]
            )
    },

    {
        "experiment":
            "R4",

        "change":
            "Optuna automated search",

        "optimizer":
            FINAL_OPTIMIZER,

        "augmentation":
            FINAL_AUGMENTATION,

        "weight_decay":
            FINAL_WEIGHT_DECAY,

        "best_epoch":
            BEST_OPTUNA_EPOCH,

        # AUC comes from the reconstructed selected R4 configuration.
        "validation_auc":
            float(
                r4_selected_row[
                    "validation_auc"
                ]
            ),

        # F1 is the Optuna model-selection objective.
        "validation_f1":
            float(
                study.best_value
            )
    }

])


display(
    resnet50_development_comparison
)


resnet50_development_comparison.to_csv(
    EXPERIMENT_FOLDER / "ResNet50_development_comparison.csv",
    index=False
)


print(
    "Selected final classification threshold:",
    f"{FINAL_CLASSIFICATION_THRESHOLD:.3f}"
)

## 7. Final model fitting

Once the model configuration, hyperparameters, training duration and classification threshold have been selected, all eight development sites are combined for final model fitting. This allows the final classifier to use all labelled development data while preserving the two held-out sites for an unbiased final evaluation.

No further model-selection decisions are made after this point.

In [ ]:
# Cell 35: Final development training dataset

# All eight development sites are now used for final training.
# The two held-out test sites remain completely excluded.

final_train_df = (
    dataset_df[
        dataset_df["site"].isin(
            DEVELOPMENT_SITES
        )
    ]
    .reset_index(drop=True)
)

final_test_df = (
    dataset_df[
        dataset_df["site"].isin(
            TEST_SITES
        )
    ]
    .reset_index(drop=True)
)


assert len(final_train_df) == 3200
assert len(final_test_df) == 800

assert set(
    final_train_df["site"]
).isdisjoint(
    set(final_test_df["site"])
)


# R4 selected augmentation = False.
# Therefore the same deterministic ImageNet preprocessing
# used in R0 is applied during final training.

final_train_dataset = CulvertResNetDataset(
    final_train_df,
    transform=R0_TRANSFORM
)

final_test_dataset = CulvertResNetDataset(
    final_test_df,
    transform=R0_TRANSFORM
)


final_train_loader = DataLoader(
    final_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

final_test_loader = DataLoader(
    final_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("Final training images:", len(final_train_df))
print("Held-out test images:", len(final_test_df))

print("\nFinal training sites:")
for site in DEVELOPMENT_SITES:
    print(" ", site)

print("\nHeld-out test sites:")
for site in TEST_SITES:
    print(" ", site)

print("\nFinal training class counts:")
print(
    final_train_df["label"].value_counts()
)

print("\nTest class counts:")
print(
    final_test_df["label"].value_counts()
)

In [ ]:
# Cell 36: Build final selected ResNet50

# Final configuration selected during development:
#
# Architecture: ResNet50
# Weights: torchvision ImageNet1K_V2
# Backbone: frozen
# Classification head: trainable
# Optimizer: AdamW
# Learning rate: 0.001
# Weight decay: Optuna-selected 0.000314288...
# Augmentation: False
# Training duration: 18 epochs
#
# No further hyperparameter selection is performed.

torch.manual_seed(
    RANDOM_SEED
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


final_model = models.resnet50(
    weights=ResNet50_Weights.IMAGENET1K_V2
)


for parameter in final_model.parameters():
    parameter.requires_grad = False


in_features = (
    final_model.fc.in_features
)

final_model.fc = nn.Linear(
    in_features,
    2
)

final_model = final_model.to(
    DEVICE
)


final_criterion = (
    nn.CrossEntropyLoss()
)


# PyTorch AdamW optimizer.
# Hyperparameter values were fixed during
# the preceding development-stage Optuna search.

final_optimizer = optim.AdamW(
    final_model.fc.parameters(),
    lr=0.001,
    weight_decay=FINAL_WEIGHT_DECAY
)


FINAL_EPOCHS = (
    BEST_OPTUNA_EPOCH
)


print("Final ResNet50 configuration")
print("Optimizer: AdamW")
print("Learning rate: 0.001")
print(
    "Weight decay:",
    FINAL_WEIGHT_DECAY
)
print(
    "Augmentation:",
    FINAL_AUGMENTATION
)
print(
    "Epochs:",
    FINAL_EPOCHS
)

In [ ]:
# Cell 37: Train final ResNet50 on all development sites

final_training_history = []

start_time = time.time()


for epoch in range(
    1,
    FINAL_EPOCHS + 1
):

    train_loss, train_f1 = train_one_epoch(
        final_model,
        final_train_loader,
        final_criterion,
        final_optimizer
    )


    final_training_history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_f1":
            train_f1
    })


    print(
        f"Epoch {epoch:02d}/{FINAL_EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"train F1 {train_f1:.4f}"
    )


final_training_seconds = (
    time.time()
    - start_time
)


final_training_history_df = (
    pd.DataFrame(
        final_training_history
    )
)


print("\nFinal training complete")

print(
    "Training time:",
    f"{final_training_seconds / 60:.2f} minutes"
)

In [ ]:
# Cell 38: Save final trained model

FINAL_MODEL_PATH = (
    EXPERIMENT_FOLDER / "ResNet50_final_selected_model.pt"
)


torch.save(
    {
        "model_state_dict":
            final_model.state_dict(),

        "architecture":
            "ResNet50",

        "pretrained_weights":
            "ImageNet1K_V2",

        "optimizer":
            "AdamW",

        "learning_rate":
            0.001,

        "weight_decay":
            FINAL_WEIGHT_DECAY,

        "augmentation":
            FINAL_AUGMENTATION,

        "epochs":
            FINAL_EPOCHS,

        # Selected using development validation data only
        "classification_threshold":
            FINAL_CLASSIFICATION_THRESHOLD,

        "development_sites":
            DEVELOPMENT_SITES,

        "held_out_test_sites":
            TEST_SITES,

        "random_seed":
            RANDOM_SEED
    },

    FINAL_MODEL_PATH
)


final_training_history_df.to_csv(
    EXPERIMENT_FOLDER / "ResNet50_final_training_history.csv",
    index=False
)


print("Final model saved:")
print(FINAL_MODEL_PATH)

print(
    "Classification threshold saved:",
    f"{FINAL_CLASSIFICATION_THRESHOLD:.3f}"
)

## 8. Held-out cross-site evaluation

The final model is evaluated on **Cornwall Bude Cedar Grove** and **Brutondam**, neither of which was used for training, validation, hyperparameter search or threshold selection. Performance is reported both on the pooled 800-image held-out set and separately by site.

ROC-AUC measures ranking/discrimination independently of the fixed classification threshold. Accuracy, precision, recall and F1 describe classification behaviour at the development-selected threshold. Confusion matrices make the balance between false alarms and missed blockages explicit.

In [ ]:
# Cell 39: Final held-out evaluation

# The classification threshold was selected using
# development validation data only.
# It is fixed before evaluation on the held-out test sites.

final_model.eval()

all_targets = []
all_predictions = []
all_scores = []
all_sites = []
all_paths = []


with torch.no_grad():

    for images, labels, sites, paths in final_test_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        logits = final_model(
            images
        )

        # Probability assigned to the blocked class
        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]


        # Apply the classification threshold selected
        # from development validation data.
        predictions = (
            probabilities
            >= FINAL_CLASSIFICATION_THRESHOLD
        ).long()


        all_targets.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_scores.extend(
            probabilities.cpu().numpy()
        )

        all_sites.extend(
            sites
        )

        all_paths.extend(
            paths
        )


final_results_df = pd.DataFrame({

    "site":
        all_sites,

    "image_path":
        all_paths,

    "true_label":
        all_targets,

    "prediction":
        all_predictions,

    "blocked_probability":
        all_scores
})


pooled_metrics = {

    "auc":
        roc_auc_score(
            final_results_df["true_label"],
            final_results_df["blocked_probability"]
        ),

    "accuracy":
        accuracy_score(
            final_results_df["true_label"],
            final_results_df["prediction"]
        ),

    "precision":
        precision_score(
            final_results_df["true_label"],
            final_results_df["prediction"],
            zero_division=0
        ),

    "recall":
        recall_score(
            final_results_df["true_label"],
            final_results_df["prediction"],
            zero_division=0
        ),

    "f1":
        f1_score(
            final_results_df["true_label"],
            final_results_df["prediction"],
            zero_division=0
        )
}


print(
    "Final classification threshold:",
    f"{FINAL_CLASSIFICATION_THRESHOLD:.3f}"
)

print(
    "\nFinal pooled held-out performance"
)

for metric, value in pooled_metrics.items():

    print(
        f"{metric}: {value:.4f}"
    )


final_results_df.to_csv(
    EXPERIMENT_FOLDER / "ResNet50_final_predictions.csv",
    index=False
)

In [ ]:
# Cell 40: Final per-site held-out metrics

site_rows = []


for site in TEST_SITES:

    site_df = final_results_df[
        final_results_df["site"] == site
    ]

    site_rows.append({

        "site":
            site,

        "auc":
            roc_auc_score(
                site_df["true_label"],
                site_df["blocked_probability"]
            ),

        "accuracy":
            accuracy_score(
                site_df["true_label"],
                site_df["prediction"]
            ),

        "precision":
            precision_score(
                site_df["true_label"],
                site_df["prediction"],
                zero_division=0
            ),

        "recall":
            recall_score(
                site_df["true_label"],
                site_df["prediction"],
                zero_division=0
            ),

        "f1":
            f1_score(
                site_df["true_label"],
                site_df["prediction"],
                zero_division=0
            )
    })


final_site_metrics_df = pd.DataFrame(
    site_rows
)


display(
    final_site_metrics_df
)


final_site_metrics_df.to_csv(
    EXPERIMENT_FOLDER / "ResNet50_final_per_site_metrics.csv",
    index=False
)

In [ ]:
# Cell 41: Final held-out confusion matrices

import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix
)


SITE_DISPLAY_NAMES = {
    "Cornwall_BudeCedarGrove":
        "Cornwall Bude Cedar Grove",

    "sites_brutondam_cam1":
        "Brutondam"
}


plot_groups = [
    (
        "Pooled",
        final_results_df
    )
]


for site in TEST_SITES:

    plot_groups.append(
        (
            SITE_DISPLAY_NAMES[site],

            final_results_df[
                final_results_df["site"] == site
            ]
        )
    )


fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4.5)
)


for axis, (title, result_df) in zip(
    axes,
    plot_groups
):

    cm = confusion_matrix(
        result_df["true_label"],
        result_df["prediction"],
        labels=[0, 1]
    )

    display_cm = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "clear",
            "blocked"
        ]
    )

    display_cm.plot(
        ax=axis,
        cmap="Blues",
        colorbar=False,
        values_format="d"
    )

    axis.set_title(
        title
    )


fig.suptitle(
    "Final ResNet50 performance on unseen test sites",
    fontsize=14
)

plt.tight_layout()

plt.savefig(
    EXPERIMENT_FOLDER / "ResNet50_final_confusion_matrices.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Reported final ResNet50 classification performance

In the dissertation run, the final ResNet50 achieved a pooled accuracy of 0.771 and F1 score of 0.751 across the two unseen test sites using the classification threshold of 0.840 selected from the development validation data. Of the 400 blocked images, 276 were correctly identified, corresponding to a recall of 0.690. Of the 400 clear images, 341 were correctly classified.

Performance remained site dependent. At Cornwall Bude Cedar Grove, the model achieved high precision (0.854) but lower recall (0.615), correctly identifying 123 of 200 blocked images while misclassifying 77 as clear. At Brutondam, performance was more balanced, with precision of 0.801 and recall of 0.765, correctly identifying 153 of 200 blocked images.

The results indicate that the model transferred to both camera sites that were excluded from training, although differences in precision and recall remained between sites. This suggests that classification behaviour continues to be influenced by site-specific visual conditions despite the model retaining useful discrimination on unseen sites.

These values are retained as the reported dissertation result. Re-running the notebook will regenerate the metrics from the repository data and current software environment.


In [ ]:
# Cell 42: Final held-out ROC curves

from sklearn.metrics import (
    roc_curve,
    roc_auc_score
)

fig, axis = plt.subplots(
    figsize=(8, 6)
)


# Pooled ROC
pooled_fpr, pooled_tpr, _ = roc_curve(
    final_results_df["true_label"],
    final_results_df["blocked_probability"]
)

pooled_auc = roc_auc_score(
    final_results_df["true_label"],
    final_results_df["blocked_probability"]
)

axis.plot(
    pooled_fpr,
    pooled_tpr,
    linewidth=2,
    label=f"Pooled (AUC = {pooled_auc:.3f})"
)


# Per-site ROC
for site in TEST_SITES:

    site_df = final_results_df[
        final_results_df["site"] == site
    ]

    site_fpr, site_tpr, _ = roc_curve(
        site_df["true_label"],
        site_df["blocked_probability"]
    )

    site_auc = roc_auc_score(
        site_df["true_label"],
        site_df["blocked_probability"]
    )

    # Human-readable site names for reporting
    display_name = SITE_DISPLAY_NAMES[
        site
    ]

    axis.plot(
        site_fpr,
        site_tpr,
        linewidth=2,
        label=(
            f"{display_name} "
            f"(AUC = {site_auc:.3f})"
        )
    )


# Chance reference
axis.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Chance"
)


axis.set_xlabel(
    "False positive rate"
)

axis.set_ylabel(
    "True positive rate"
)

axis.set_title(
    "ResNet50 performance on unseen test sites"
)

axis.legend(
    loc="lower right"
)

fig.tight_layout()


plt.savefig(
    EXPERIMENT_FOLDER / "ResNet50_final_ROC.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Reported final ResNet50 ROC performance

In the dissertation run, the final ResNet50 achieved a pooled ROC-AUC of 0.820 across the two unseen test sites. Site-specific performance was similar, with an AUC of 0.819 at Cornwall Bude Cedar Grove and 0.832 at Brutondam.

The similarity between the site-specific AUC values indicates that the model retained a relatively consistent ability to discriminate between clear and blocked images across both unseen camera sites. However, the differences in precision and recall observed at the fixed classification threshold show that similar discriminatory performance does not necessarily result in identical classification behaviour across sites.

These results indicate that the model retained useful discriminatory information when transferred to camera sites that were excluded from training, while some site-specific variation remained in the final classifications.

## 9. Reproducibility artefacts

The final cells save the trained model state, training history, image-level held-out predictions, per-site metrics and a compact JSON summary.

In [ ]:
# Cell 43: Save final ResNet50 results

import json


FINAL_RESULTS_PATH = (
    EXPERIMENT_FOLDER / "ResNet50_final_results.json"
)


final_results = {

    "model": {
        "architecture": "ResNet50",
        "pretrained_weights": "ImageNet1K_V2",
        "backbone": "frozen",
        "batchnorm_statistics": "fixed",
        "trainable_layer": "final_fc_layer",
        "classes": 2
    },


    "training": {
        "optimizer": FINAL_OPTIMIZER,
        "learning_rate": 0.001,
        "weight_decay": float(FINAL_WEIGHT_DECAY),
        "augmentation": bool(FINAL_AUGMENTATION),
        "epochs": int(FINAL_EPOCHS),
        "batch_size": int(BATCH_SIZE),
        "random_seed": int(RANDOM_SEED)
    },


    "selection": {
        "optuna_trial": int(best_trial.number),
        "best_validation_f1": float(study.best_value),
        "best_epoch": int(BEST_OPTUNA_EPOCH),
        "classification_threshold": float(
            FINAL_CLASSIFICATION_THRESHOLD
        ),
        "threshold_selected_using": "validation_f1"
    },


    "data": {
        "development_sites": list(DEVELOPMENT_SITES),
        "training_sites": list(TRAIN_SITES),
        "validation_sites": list(VALIDATION_SITES),
        "test_sites": list(TEST_SITES),
        "final_training_images": int(len(final_train_df)),
        "final_test_images": int(len(final_test_df))
    },


    "test_results": {
        "auc": float(pooled_metrics["auc"]),
        "accuracy": float(pooled_metrics["accuracy"]),
        "precision": float(pooled_metrics["precision"]),
        "recall": float(pooled_metrics["recall"]),
        "f1": float(pooled_metrics["f1"])
    },


    "test_results_by_site":
        final_site_metrics_df.to_dict(
            orient="records"
        )
}


with open(
    FINAL_RESULTS_PATH,
    "w"
) as file:

    json.dump(
        final_results,
        file,
        indent=4
    )


print("Final ResNet50 results saved:")
print(FINAL_RESULTS_PATH)